# 04. ASSOCIACAO DOS ACIDENTES AS VIAS

ESTE E O NOTEBOOK QUE ENTREGA O OBJETIVO DO PROJETO: **CADA ACIDENTE VINCULADO A UMA VIA DO BANCO**.

DEPENDE DE DOIS NOTEBOOKS:

- **02**, QUE CRIA A TABELA `vias_processadas` COM O `via_id` DE CADA VIA.
- **03**, QUE DEIXA OS ENDERECOS PADRONIZADOS EM `acidentes_revisado`.

## A IDEIA CENTRAL, EM CINCO PASSOS

1. MARCAR AS LINHAS QUE PRECISAM DE VIA E RESOLVER DE CARA AS QUE NAO TEM ENDERECO.
2. PEGAR OS PARES DISTINTOS DE ENDERECO E BAIRRO JA PADRONIZADOS.
3. PARA CADA PAR: BUSCAR AS VIAS CANDIDATAS E PEDIR A IA QUE ESCOLHA UMA.
4. ESPALHAR O `via_id` ESCOLHIDO PARA TODAS AS LINHAS DAQUELE PAR.
5. REGISTRAR A COBERTURA, PARA PODER RETOMAR DEPOIS.

## COMO A ESCOLHA ACONTECE

A BUSCA NAO DECIDE SOZINHA. ELA APENAS **PROPOE** AS VIAS MAIS PARECIDAS, COMBINANDO SEMELHANCA DE TEXTO COM SEMELHANCA DE SIGNIFICADO. QUEM DECIDE E A IA, OLHANDO A LISTA.

```text
ENDERECO PADRONIZADO         BUSCA PROPOE (6)              A IA ESCOLHE
'Avenida Independencia'  ->  1. Avenida Independencia  ->  id 1
  bairro 'Cidade Livre'      2. Avenida Independente
                             3. Rua Independencia
                             ...
```

## LEIA ISTO ANTES DE CONFIAR NO RESULTADO

A DECISAO E GUIADA PELO **NOME DA VIA**. ISSO FUNCIONA BEM QUANDO O NOME E UNICO, E FUNCIONA MAL QUANDO NAO E.

EM APARECIDA DE GOIANIA, CERCA DE **35 POR CENTO DAS VIAS COMPARTILHAM NOME COM OUTRA VIA DA MESMA CIDADE**. EXISTEM 23 RUAS CHAMADAS `Rua 1`, EM SETORES DIFERENTES.

O BAIRRO SERVE PARA DESEMPATAR, MAS ELE VEM DO OPENSTREETMAP E ESTA VAZIO EM CERCA DE 38 POR CENTO DAS VIAS QUE COLIDEM. QUANDO NAO HA BAIRRO PARA DESEMPATAR, A INSTRUCAO MANDA FICAR COM A PRIMEIRA CANDIDATA, QUE ENTRE VARIAS HOMONIMAS E PRATICAMENTE ARBITRARIA.

O RESULTADO SAI MARCADO COMO `ok`, COM UMA NOTA ALTA, E NAO HA COMO DISTINGUIR ESSE CASO DE UM ACERTO REAL SO OLHANDO A TABELA.

**A ETAPA 9 MEDE QUANTO DO SEU RESULTADO CAIU NESSA SITUACAO.** OLHE ESSE NUMERO ANTES DE USAR OS DADOS EM QUALQUER ANALISE.

## 1. IMPORTACAO DAS BIBLIOTECAS E CONFIGURACOES

EXPLICACAO DOS VALORES:

- `ANO` E `CODIGO_IBGE`: O RECORTE QUE SERA PROCESSADO. USE O MESMO QUE VOCE REVISOU NO NOTEBOOK 03.
- `TOP_N`: QUANTAS VIAS CANDIDATAS A BUSCA PROPOE PARA CADA ENDERECO. A INSTRUCAO ENVIADA A IA USA ESTE MESMO NUMERO, ENTAO OS DOIS NUNCA FICAM DESALINHADOS.
- `TAMANHO_LOTE`: QUANTOS PARES, COM SUAS CANDIDATAS, VAO EM CADA CHAMADA. E MENOR QUE O DO NOTEBOOK 03 PORQUE CADA ITEM AQUI CARREGA UMA LISTA INTEIRA DE CANDIDATAS.
- `PAUSA_ENTRE_CHAMADAS`: SEGUNDOS DE PAUSA ENTRE UMA CHAMADA E A SEGUINTE.
- `TENTATIVAS`: QUANTAS VEZES TENTAR UM LOTE ANTES DE DEIXAR PARA A PROXIMA EXECUCAO.
- `MAX_TOKENS_RESPOSTA`: TETO DE TOKENS DA RESPOSTA, PARA ELA NAO VIR TRUNCADA.
- `PENSAMENTO_LIGADO`: SE O MODELO RACIOCINA ANTES DE ESCOLHER. VEM DESLIGADO. EM TESTE COM CASOS DIFICEIS, INCLUINDO NOME REPETIDO E DESEMPATE POR BAIRRO, A ESCOLHA FOI A MESMA COM E SEM RACIOCINIO, GASTANDO VINTE VEZES MAIS TOKENS DE SAIDA COM ELE LIGADO.

SOBRE O `TOP_N`: AUMENTAR ESSE NUMERO AJUDA POUCO QUANDO O PROBLEMA E NOME REPETIDO. SE HA 23 RUAS CHAMADAS `Rua 1`, MOSTRAR 10 EM VEZ DE 6 SO AUMENTA O NUMERO DE OPCOES IGUALMENTE PLAUSIVEIS.

In [13]:
# JSON MONTA A ENTRADA E LE A RESPOSTA DA IA.
import json

# SQLITE3 E O BANCO LOCAL DO PROJETO.
import sqlite3

# SYS E USADO PARA GARANTIR QUE O PYTHON ENCONTRE OS ARQUIVOS config.py E busca.py.
import sys

# TIME E USADO PARA AS PAUSAS ENTRE CHAMADAS.
import time

# DATETIME REGISTRA QUANDO CADA COISA FOI FEITA.
from datetime import datetime, timezone

# PATH AJUDA A TRABALHAR COM CAMINHOS DE ARQUIVO.
from pathlib import Path

# PANDAS E USADO PARA MOSTRAR AS TABELAS DE CONFERENCIA.
import pandas as pd

# CLIENTE DA API. O DEEPSEEK FALA O PROTOCOLO DA OPENAI, ENTAO A BIBLIOTECA
# INSTALADA E A `openai` MESMO QUE O MODELO SEJA DEEPSEEK.
from openai import OpenAI

# GARANTE QUE A PASTA projeto_v2 ESTEJA NO CAMINHO DE IMPORTACAO,
# INDEPENDENTE DE ONDE O JUPYTER FOI ABERTO.
for _candidata in (Path.cwd(), Path.cwd() / "projeto_v2", Path.cwd().parent):
    if (_candidata / "config.py").exists():
        sys.path.insert(0, str(_candidata))
        break

# CONFIGURACAO COMPARTILHADA E O MOTOR DE BUSCA DE VIAS.
import config

# O QUE SERA PROCESSADO: MUNICIPIO, ANO E AS DEMAIS ESCOLHAS. TODAS ELAS VIVEM NO
# parametros.py -- E O UNICO ARQUIVO QUE VOCE EDITA ANTES DE RODAR.
import parametros
from busca import MotorBusca, carregar_vias

# =========================
# PARAMETROS DESTA EXECUCAO
# =========================

# ANO E MUNICIPIO QUE SERAO PROCESSADOS. VEM DO MESMO LUGAR QUE O NOTEBOOK 03 LE,
# ENTAO OS DOIS NAO PODEM MAIS PROCESSAR RECORTES DIFERENTES POR DESCUIDO.
ANO = parametros.ANO
CODIGO_IBGE = parametros.CODIGO_IBGE

# =========================
# CONFIGURACOES DA ASSOCIACAO
# =========================

# QUANTAS VIAS CANDIDATAS A BUSCA PROPOE PARA CADA ENDERECO.
TOP_N = parametros.TOP_N

# QUANTOS PARES, COM SUAS CANDIDATAS, POR CHAMADA A IA.
TAMANHO_LOTE = parametros.TAMANHO_LOTE_ASSOCIACAO

# TETO DE TOKENS DA RESPOSTA. O MODO JSON DA API TRUNCA A RESPOSTA SEM AVISAR SE
# ESTE VALOR FOR CURTO, E RESPOSTA TRUNCADA FAZ O LOTE FALHAR (VEJA A ETAPA 3).
MAX_TOKENS_RESPOSTA = parametros.MAX_TOKENS_RESPOSTA

# O deepseek-v4-flash RACIOCINA ANTES DE RESPONDER POR PADRAO. AQUI ISSO E DESPERDICIO:
# EM TESTE COM OS MESMOS ENDERECOS, A SAIDA FOI IDENTICA COM E SEM RACIOCINIO, E COM
# ELE LIGADO O GASTO DE TOKENS DE SAIDA FOI CERCA DE DEZ VEZES MAIOR.
# COLOQUE True SE QUISER COMPARAR VOCE MESMO.
PENSAMENTO_LIGADO = parametros.PENSAMENTO_LIGADO

# `extra_body` LEVA PARAMETROS QUE SAO DO DEEPSEEK E NAO EXISTEM NA API DA OPENAI.
CORPO_EXTRA = parametros.CORPO_EXTRA

# SEGUNDOS DE PAUSA ENTRE UMA CHAMADA E A SEGUINTE.
PAUSA_ENTRE_CHAMADAS = parametros.PAUSA_ENTRE_CHAMADAS

# QUANTAS VEZES TENTAR UM LOTE ANTES DE DEIXAR PARA A PROXIMA EXECUCAO.
TENTATIVAS = parametros.TENTATIVAS

# SEPARADOR INVISIVEL, USADO SO PARA MONTAR A CHAVE DO DICIONARIO.
SEPARADOR_CHAVE = "\u001f"

# MOSTRA A CONFIGURACAO ATIVA PARA CONFERENCIA.
config.resumo()
print()
parametros.resumo()
print(f"CANDIDATAS POR VIA: {TOP_N}")

# SEM CHAVE NAO HA COMO CHAMAR A IA.
if not config.CHAVE_IA:
    raise RuntimeError("DEEPSEEK_API_KEY AUSENTE. DEFINA A CHAVE NO ARQUIVO .env DA RAIZ DO REPOSITORIO.")

BANCO DO V2       : C:\Users\fabio\Documents\GitHub\dash-sinistros-renaest\projeto_v2\data\db\db_main.db
  EXISTE?         : SIM
PASTA TEMP        : C:\Users\fabio\Documents\GitHub\dash-sinistros-renaest\projeto_v2\data\temp
PASTA CACHE       : C:\Users\fabio\Documents\GitHub\dash-sinistros-renaest\projeto_v2\data\cache
CHAVE DA IA       : DEFINIDA
MODELO DA IA      : deepseek-v4-flash
URL DA IA         : https://api.deepseek.com
EMBEDDINGS        : LIGADOS
MODELO EMBEDDINGS : intfloat/multilingual-e5-small

RECORTE: ANO 2024, MUNICIPIO 5201405
CANDIDATAS POR ENDERECO: 6


## 2. A INSTRUCAO ENVIADA A IA

ESTA INSTRUCAO DECIDE A QUALIDADE DE TODO O RESULTADO. VALE LER COM CALMA.

O PONTO MAIS DELICADO E COMO ELA TRATA O BAIRRO.

O BAIRRO DA CANDIDATA VEM DO OPENSTREETMAP E O BAIRRO DO ACIDENTE VEM DO BOLETIM DE OCORRENCIA. SAO FONTES DIFERENTES, E A DO OPENSTREETMAP E INCOMPLETA. ENTAO ACONTECE MUITO DE A VIA SER A CERTA E O BAIRRO NAO BATER.

SE A INSTRUCAO DEIXASSE O MODELO RESPONDER `null` NESSES CASOS, PERDERIAMOS UMA MONTANHA DE ASSOCIACOES CORRETAS. POR ISSO ELA E EXPLICITA: **BAIRRO QUE NAO BATE NAO DESQUALIFICA. QUEM DECIDE E O NOME.** O BAIRRO SO ENTRA PARA DESEMPATAR.

REPARE TAMBEM QUE O NUMERO DE CANDIDATAS NO TEXTO VEM DA VARIAVEL `TOP_N`. NA VERSAO ANTERIOR ESSE NUMERO ESTAVA ESCRITO A MAO NA INSTRUCAO E TINHA FICADO DESATUALIZADO: O TEXTO DIZIA 10 ENQUANTO O CODIGO ENVIAVA 6.

A RESPOSTA VEM COMO UM OBJETO JSON COM A CHAVE `resultados`, E O ARRAY DAS ESCOLHAS DENTRO DELE. ISSO E EXIGENCIA DA API: NO MODO JSON DELA A RESPOSTA E SEMPRE UM OBJETO, E A INSTRUCAO PRECISA CONTER A PALAVRA `json` E UM EXEMPLO DA SAIDA. NAO REMOVA O EXEMPLO.

In [14]:
# O NUMERO DE CANDIDATAS VEM DA VARIAVEL, NUNCA ESCRITO A MAO.
# ASSIM O TEXTO E O CODIGO NAO PODEM DIVERGIR.
INSTRUCAO = f"""Voce associa cada endereco de acidente de transito a UMA via de uma lista de candidatas.
A entrada e uma LISTA de itens; cada item tem a consulta (via + bairro do acidente) e ate {TOP_N}
vias candidatas, cada uma com um 'id' (numero), o nome da via e os bairros por onde ela passa.
As candidatas vem ordenadas por proximidade do nome (id 1 tende a ser o nome mais parecido),
mas confirme pelo conteudo: o id 1 nem sempre e a via certa.
ATENCAO sobre o bairro: a lista de bairros de cada candidata vem de outra fonte e e
INCOMPLETA; com frequencia NAO inclui o bairro do acidente mesmo quando a via e a correta.
DECIDA NESTA ORDEM:
1. Escolha pelo NOME: a candidata cujo nome e a MESMA via da consulta (mesmo logradouro).
   Ignore diferencas de acento, caixa, abreviacao ou caractere estranho no nome (e a mesma via).
2. So use o bairro para DESEMPATAR quando DUAS OU MAIS candidatas tiverem nome igualmente bom:
   prefira a que inclui o bairro do acidente. Se NENHUMA incluir, fique com o de menor 'id'.
3. NUNCA responda null so porque o bairro da candidata difere do bairro do acidente: bairro
   que nao bate NAO desqualifica. Quem decide e o nome.
4. NAO invente ids: use somente um dos ids oferecidos naquele item.
5. Responda null APENAS quando NENHUMA candidata tiver o mesmo NOME de via da consulta.
Responda APENAS um objeto json com a chave "resultados", cujo valor e um array na MESMA
ORDEM e MESMO TAMANHO da entrada, com objetos {{"escolhido": <int|null>}}.
Exemplo de saida json valida: {{"resultados": [{{"escolhido": 1}}, {{"escolhido": null}}]}}
ENTRADA:
"""

print(f"INSTRUCAO CARREGADA: {len(INSTRUCAO)} CARACTERES.")
print(f"O TEXTO ANUNCIA {TOP_N} CANDIDATAS, QUE E EXATAMENTE O QUE SERA ENVIADO.")

INSTRUCAO CARREGADA: 1560 CARACTERES.
O TEXTO ANUNCIA 6 CANDIDATAS, QUE E EXATAMENTE O QUE SERA ENVIADO.


## 3. FUNCOES AUXILIARES

AQUI ESTAO TODAS AS FUNCOES USADAS PELAS ETAPAS SEGUINTES.

### O ATALHO DOS ENDERECOS SEM LOGRADOURO

O NOTEBOOK 03 MARCA COMO REVISADAS, COM O CAMPO PADRONIZADO VAZIO, AS LINHAS CUJO ENDERECO ORIGINAL ERA `S/N`, `NAO INFORMADO` OU COISA PARECIDA.

ESSAS LINHAS ESTAO REVISADAS, MAS NAO TEM O QUE PROCURAR. A FUNCAO `marcar_sem_endereco` RESOLVE TODAS ELAS DE UMA VEZ, ANTES DE QUALQUER BUSCA, MARCANDO COMO `sem_via`.

SEM ESSE ATALHO, ELAS IRIAM PARA A BUSCA COM O NOME VAZIO, VOLTARIAM COM SEIS CANDIDATAS ALEATORIAS E AINDA GASTARIAM CHAMADA DE IA PARA O MODELO CONCLUIR O OBVIO.

### OS TRES ESTADOS POSSIVEIS DE UMA LINHA

```text
'ok'       -> UMA VIA FOI ESCOLHIDA. O CAMPO via_id_associada ESTA PREENCHIDO.
'sem_via'  -> RESOLVIDO, MAS SEM VIA. OU O ENDERECO ERA VAZIO, OU NENHUMA
              CANDIDATA TINHA O MESMO NOME.
'pendente' -> AINDA NAO DECIDIDO. UMA CHAMADA FALHOU. REEXECUTE O NOTEBOOK.
```

In [15]:
def agora():
    """HORARIO ATUAL EM TEXTO, PARA REGISTRAR QUANDO ALGO FOI FEITO."""
    return datetime.now(timezone.utc).isoformat()


def chave(end, bairro):
    """MONTA A CHAVE UNICA DE UM PAR. UM VALOR NULO VIRA TEXTO VAZIO."""
    return f"{end or ''}{SEPARADOR_CHAVE}{bairro or ''}"


def conectar():
    """ABRE A CONEXAO COM O BANCO DO PROJETO."""
    return sqlite3.connect(str(config.BANCO))


def colunas_existentes(conn, tabela):
    """LISTA OS NOMES DE COLUNA DE UMA TABELA, PARA SABER O QUE JA EXISTE."""
    return {linha[1] for linha in conn.execute(f"PRAGMA table_info({tabela})")}


def criar_tabelas(conn):
    """ACRESCENTA AS COLUNAS DE ASSOCIACAO E CRIA AS TABELAS DE APOIO."""
    existentes = colunas_existentes(conn, "acidentes_revisado")

    if not existentes:
        raise RuntimeError(
            "A TABELA 'acidentes_revisado' NAO EXISTE.\n"
            "RODE O NOTEBOOK 03_revisao_enderecos.ipynb PRIMEIRO."
        )

    # COLUNAS NOVAS NA TABELA DE RESULTADO, ACRESCENTADAS SEM DESTRUIR O QUE EXISTE.
    novas = {
        "via_id_associada": "TEXT",
        "nome_via_associada": "TEXT",
        "assoc_score": "REAL",     # NOTA DA BUSCA (0 A 1) DA CANDIDATA ESCOLHIDA
        "assoc_status": "TEXT",    # 'pendente' | 'ok' | 'sem_via'
        "assoc_modelo": "TEXT",
        "assoc_em": "TEXT",
    }

    for nome, tipo in novas.items():
        if nome not in existentes:
            conn.execute(f"ALTER TABLE acidentes_revisado ADD COLUMN {nome} {tipo}")

    conn.execute(
        f"CREATE INDEX IF NOT EXISTS idx_assoc_slice "
        f"ON acidentes_revisado({config.COLUNA_ANO}, {config.COLUNA_MUNICIPIO}, assoc_status)"
    )

    # DICIONARIO REUSAVEL: UMA LINHA POR PAR PADRONIZADO DISTINTO.
    # A COLUNA candidatos GUARDA A LISTA QUE FOI OFERECIDA A IA. E O QUE
    # PERMITE AUDITAR UMA DECISAO DEPOIS, SEM REFAZER A BUSCA.
    conn.execute("""
        CREATE TABLE IF NOT EXISTS associacao_vias (
            chave TEXT PRIMARY KEY,
            end_padronizado TEXT,
            bairro_padronizado TEXT,
            via_id TEXT,
            nome_via TEXT,
            score REAL,
            candidatos TEXT,
            modelo TEXT,
            criado_em TEXT
        )
    """)

    # MAPA DE PROGRESSO POR ANO E MUNICIPIO.
    conn.execute("""
        CREATE TABLE IF NOT EXISTS associacao_cobertura (
            ano TEXT,
            codigo_ibge TEXT,
            status TEXT,
            total_linhas INTEGER,
            distintos INTEGER,
            novos_ia INTEGER,
            reaproveitados INTEGER,
            sem_via INTEGER,
            iniciado_em TEXT,
            concluido_em TEXT,
            PRIMARY KEY (ano, codigo_ibge)
        )
    """)

    conn.commit()


def marcar_sem_endereco(conn, ano, ibge):
    """PASSO 1a: RESOLVE DE UMA VEZ AS LINHAS QUE NAO TEM ENDERECO PARA PROCURAR.

    SAO AS LINHAS QUE O NOTEBOOK 03 REVISOU MAS DEIXOU COM O PADRONIZADO VAZIO,
    PORQUE O ENDERECO ORIGINAL ERA 'S/N', 'NAO INFORMADO' OU EQUIVALENTE.
    """
    cursor = conn.execute(
        f"UPDATE acidentes_revisado "
        f"SET assoc_status='sem_via', assoc_modelo=NULL, assoc_em=? "
        f"WHERE {config.COLUNA_ANO}=? AND {config.COLUNA_MUNICIPIO}=? "
        f"  AND revisao_status='ok' AND assoc_status IS NULL "
        f"  AND (end_acidente_padronizado IS NULL OR end_acidente_padronizado='')",
        (agora(), ano, ibge),
    )
    conn.commit()
    return cursor.rowcount


def marcar_pendentes(conn, ano, ibge):
    """PASSO 1b: MARCA COMO PENDENTE AS LINHAS REVISADAS QUE AINDA PRECISAM DE VIA."""
    cursor = conn.execute(
        f"UPDATE acidentes_revisado SET assoc_status='pendente' "
        f"WHERE {config.COLUNA_ANO}=? AND {config.COLUNA_MUNICIPIO}=? "
        f"  AND revisao_status='ok' AND assoc_status IS NULL "
        f"  AND end_acidente_padronizado IS NOT NULL AND end_acidente_padronizado<>''",
        (ano, ibge),
    )
    conn.commit()
    return cursor.rowcount


def pares_pendentes(conn, ano, ibge):
    """PASSO 2: PEGA OS PARES PADRONIZADOS DISTINTOS AINDA NAO ASSOCIADOS."""
    return conn.execute(
        f"SELECT DISTINCT end_acidente_padronizado, bairro_acidente_padronizado "
        f"FROM acidentes_revisado "
        f"WHERE {config.COLUNA_ANO}=? AND {config.COLUNA_MUNICIPIO}=? AND assoc_status='pendente'",
        (ano, ibge),
    ).fetchall()


def par_conhecido(conn, end, bairro):
    """DIZ SE O PAR JA ESTA NO DICIONARIO, OU SEJA, JA FOI DECIDIDO ANTES."""
    return conn.execute(
        "SELECT 1 FROM associacao_vias WHERE chave=?", (chave(end, bairro),)
    ).fetchone() is not None


def buscar_candidatos(motor, end, bairro):
    """DEVOLVE AS TOP_N VIAS MAIS PARECIDAS COM O PAR, JA RANQUEADAS."""
    return motor.buscar(via=end or "", bairro=bairro or "", limit=TOP_N)


def extrair_itens(texto, tamanho_esperado):
    """TIRA O ARRAY DE RESULTADOS DA RESPOSTA E CONFERE QUE ELE VEIO COMPLETO.

    LEVANTA ERRO SEMPRE QUE A RESPOSTA NAO SERVE. QUEM CHAMA TRATA ISSO COMO
    CHAMADA QUE FALHOU: O LOTE FICA PENDENTE E E TENTADO DE NOVO.

    ISSO E DELIBERADO, E E A DIFERENCA MAIS IMPORTANTE EM RELACAO A VERSAO COM
    GEMINI. LA, UMA RESPOSTA CURTA OU FORA DE FORMA VIRAVA ITEM VAZIO -- E ITEM
    VAZIO, NESTE PROJETO, SIGNIFICA 'DECISAO CONCLUIDA SEM RESULTADO', QUE E
    GRAVADA NO DICIONARIO E NUNCA MAIS REVISTA. UMA RESPOSTA TRUNCADA APAGARIA
    ENDERECOS PARA SEMPRE, EM SILENCIO. AGORA ELA FALHA ALTO.

    O MODO JSON DESTA API DEVOLVE UM OBJETO, NAO UM ARRAY. POR ISSO A INSTRUCAO
    PEDE O ARRAY DENTRO DA CHAVE 'resultados'.
    """
    dados = json.loads(texto)

    if isinstance(dados, dict):
        itens = dados.get("resultados")

        # SE O MODELO INVENTOU OUTRO NOME DE CHAVE, PEGA O PRIMEIRO ARRAY QUE ACHAR.
        if not isinstance(itens, list):
            itens = next((v for v in dados.values() if isinstance(v, list)), None)
    else:
        # ARRAY NA RAIZ TAMBEM E ACEITO, CASO A API MUDE DE COMPORTAMENTO.
        itens = dados

    if not isinstance(itens, list):
        raise ValueError(f"RESPOSTA SEM ARRAY DE RESULTADOS: {texto[:200]}")

    if len(itens) != tamanho_esperado:
        raise ValueError(
            f"RESPOSTA COM {len(itens)} ITENS, ESPERADOS {tamanho_esperado}. "
            f"SE REPETIR, DIMINUA TAMANHO_LOTE OU AUMENTE MAX_TOKENS_RESPOSTA."
        )

    if not all(isinstance(item, dict) for item in itens):
        raise ValueError("RESPOSTA COM ITEM QUE NAO E OBJETO JSON.")

    return itens


def pedir_a_ia(client, modelo, lote):
    """MANDA UM LOTE COM SUAS CANDIDATAS E DEVOLVE O ID ESCOLHIDO DE CADA ITEM.

    DEVOLVE None INTEIRO SE A CHAMADA FALHOU DEPOIS DE TODAS AS TENTATIVAS.
    UM None DENTRO DA LISTA E DIFERENTE: SIGNIFICA QUE A IA DECIDIU QUE NENHUMA
    CANDIDATA SERVE, O QUE E UMA RESPOSTA VALIDA.

    RESPOSTA INCOMPLETA OU FORA DE FORMA CONTA COMO CHAMADA QUE FALHOU, NAO COMO
    UMA FILA DE null. ASSIM UM LOTE TRUNCADO NAO MARCA VARIOS PARES COMO 'sem_via'
    SEM NINGUEM TER DECIDIDO NADA SOBRE ELES.
    """
    # MONTA A ENTRADA: A CONSULTA E A LISTA DE CANDIDATAS DE CADA ITEM.
    entrada = []
    for p in lote:
        candidatos = [
            {"id": c["ranking"], "nome_via": c["nome_via"], "bairros": c["bairros"]}
            for c in p["candidatos"]
        ]
        entrada.append({
            "consulta": {"via": p["end"], "bairro": p["bairro"]},
            "candidatos": candidatos,
        })

    for tentativa in range(1, TENTATIVAS + 1):
        try:
            resposta = client.chat.completions.create(
                model=modelo,
                messages=[{"role": "user",
                           "content": INSTRUCAO + json.dumps(entrada, ensure_ascii=False)}],
                response_format={"type": "json_object"},
                temperature=0,
                max_tokens=MAX_TOKENS_RESPOSTA,
                extra_body=CORPO_EXTRA,
            )

            # CONFERE A FORMA E O TAMANHO ANTES DE OLHAR O CONTEUDO.
            respostas = extrair_itens(resposta.choices[0].message.content, len(lote))

            # ALINHA A RESPOSTA COM A ENTRADA, ITEM A ITEM, NA ORDEM.
            escolhas = []
            for item in respostas:
                valor = item.get("escolhido")
                escolhas.append(valor if isinstance(valor, int) else None)

            return escolhas

        except Exception as erro:
            if tentativa == TENTATIVAS:
                print(f"\n[ASSOCIACAO] LOTE FALHOU APOS {TENTATIVAS} TENTATIVAS: {erro}")
                print("[ASSOCIACAO] ESSES PARES FICAM PENDENTES PARA A PROXIMA EXECUCAO.")
                return None
            time.sleep(2 * tentativa)


def resolver_escolha(candidatos, id_escolhido):
    """ACHA A CANDIDATA CUJO ID BATE COM O ESCOLHIDO PELA IA."""
    # O RANKING COMECA EM 1, ENTAO ZERO E NULO SIGNIFICAM 'NENHUMA'.
    if not id_escolhido:
        return None

    for c in candidatos:
        if c["ranking"] == id_escolhido:
            return c

    # ID INVENTADO PELA IA TAMBEM VIRA 'NENHUMA'.
    return None


def salvar_no_dicionario(conn, modelo, end, bairro, via, candidatos):
    """GRAVA NO DICIONARIO A DECISAO DE UM PAR.

    A LISTA DE CANDIDATAS E GRAVADA JUNTO. E ELA QUE PERMITE, MESES DEPOIS,
    ENTENDER POR QUE A IA ESCOLHEU AQUELA VIA E NAO OUTRA.
    """
    conn.execute(
        "INSERT OR REPLACE INTO associacao_vias "
        "(chave, end_padronizado, bairro_padronizado, via_id, nome_via, score, "
        " candidatos, modelo, criado_em) "
        "VALUES (?,?,?,?,?,?,?,?,?)",
        (
            chave(end, bairro), end, bairro,
            via["via_id"] if via else None,
            via["nome_via"] if via else None,
            via["score"] if via else None,
            json.dumps(candidatos, ensure_ascii=False),
            modelo, agora(),
        ),
    )
    conn.commit()


def associar_pares(conn, motor, client, modelo, pares):
    """PASSO 3: PROCESSA OS PARES EM LOTES, BUSCANDO CANDIDATAS E PEDINDO A ESCOLHA."""
    total = len(pares)
    novos = 0
    falhados = 0

    for inicio in range(0, total, TAMANHO_LOTE):
        lote_pares = pares[inicio:inicio + TAMANHO_LOTE]

        # MONTA O LOTE SO COM OS PARES INEDITOS. OS CONHECIDOS SERAO
        # REAPROVEITADOS DIRETO DO DICIONARIO NO PASSO 4.
        lote = []
        for end, bairro in lote_pares:
            if par_conhecido(conn, end, bairro):
                continue
            lote.append({
                "end": end,
                "bairro": bairro,
                "candidatos": buscar_candidatos(motor, end, bairro),
            })

        if lote:
            escolhas = pedir_a_ia(client, modelo, lote)

            if escolhas is None:
                # A CHAMADA FALHOU: NADA E GRAVADO E OS PARES SEGUEM PENDENTES.
                falhados += len(lote)
            else:
                for par, id_escolhido in zip(lote, escolhas):
                    via = resolver_escolha(par["candidatos"], id_escolhido)
                    salvar_no_dicionario(conn, modelo, par["end"], par["bairro"],
                                         via, par["candidatos"])
                novos += len(lote)

                # RESPEITA O RITMO ANTES DA PROXIMA CHAMADA.
                time.sleep(PAUSA_ENTRE_CHAMADAS)

        processados = min(inicio + TAMANHO_LOTE, total)
        print(f"\r{processados}/{total} PARES  |  NOVOS NA IA: {novos}  |  "
              f"PENDENTES POR FALHA: {falhados}", end="")

    print()
    return novos, falhados


def espalhar_resultado(conn, modelo, ano, ibge):
    """PASSO 4: COPIA A VIA ESCOLHIDA PARA TODAS AS LINHAS DAQUELE PAR.

    PAR COM VIA VIRA 'ok'. PAR QUE A IA NAO CASOU VIRA 'sem_via'.
    PAR AINDA NAO DECIDIDO CONTINUA PENDENTE.
    """
    com_via = 0
    sem_via = 0

    for end, bairro in pares_pendentes(conn, ano, ibge):
        decisao = conn.execute(
            "SELECT via_id, nome_via, score FROM associacao_vias WHERE chave=?",
            (chave(end, bairro),),
        ).fetchone()

        # PAR AINDA NAO DECIDIDO: FICA PENDENTE PARA A PROXIMA EXECUCAO.
        if decisao is None:
            continue

        via_id, nome_via, score = decisao
        status = "sem_via" if via_id is None else "ok"

        cursor = conn.execute(
            f"UPDATE acidentes_revisado "
            f"SET via_id_associada=?, nome_via_associada=?, assoc_score=?, "
            f"    assoc_status=?, assoc_modelo=?, assoc_em=? "
            f"WHERE {config.COLUNA_ANO}=? AND {config.COLUNA_MUNICIPIO}=? "
            f"  AND assoc_status='pendente' "
            f"  AND end_acidente_padronizado IS ? AND bairro_acidente_padronizado IS ?",
            (via_id, nome_via, score, status, modelo, agora(), ano, ibge, end, bairro),
        )

        if status == "ok":
            com_via += cursor.rowcount
        else:
            sem_via += cursor.rowcount

    conn.commit()
    return com_via, sem_via


def registrar_cobertura(conn, ano, ibge, distintos, novos):
    """PASSO 5: ANOTA NA COBERTURA QUANTO FOI FEITO NESTE RECORTE."""
    def contar(condicao_extra=""):
        return conn.execute(
            f"SELECT count(*) FROM acidentes_revisado "
            f"WHERE {config.COLUNA_ANO}=? AND {config.COLUNA_MUNICIPIO}=? "
            f"  AND revisao_status='ok' {condicao_extra}",
            (ano, ibge),
        ).fetchone()[0]

    total = contar()
    sem_via = contar("AND assoc_status='sem_via'")
    pendentes = contar("AND (assoc_status='pendente' OR assoc_status IS NULL)")

    # SOMA AOS CONTADORES DE UMA EXECUCAO ANTERIOR, SE HOUVER.
    anterior = conn.execute(
        "SELECT distintos, novos_ia FROM associacao_cobertura WHERE ano=? AND codigo_ibge=?",
        (ano, ibge),
    ).fetchone()
    distintos_antes, novos_antes = anterior if anterior else (0, 0)

    distintos_total = (distintos_antes or 0) + distintos
    novos_total = (novos_antes or 0) + novos

    status = "ok" if pendentes == 0 else "parcial"

    conn.execute(
        "INSERT OR REPLACE INTO associacao_cobertura "
        "(ano, codigo_ibge, status, total_linhas, distintos, novos_ia, "
        " reaproveitados, sem_via, concluido_em) "
        "VALUES (?,?,?,?,?,?,?,?,?)",
        (ano, ibge, status, total, distintos_total, novos_total,
         distintos_total - novos_total, sem_via, agora()),
    )
    conn.commit()

    return {"total": total, "distintos": distintos_total, "novos_ia": novos_total,
            "reaproveitados": distintos_total - novos_total,
            "sem_via": sem_via, "pendentes": pendentes, "status": status}


print("FUNCOES AUXILIARES CARREGADAS.")

FUNCOES AUXILIARES CARREGADAS.


## 4. PREPARAR O BANCO E CARREGAR O MOTOR DE BUSCA

O MOTOR DE BUSCA E CARREGADO **UMA VEZ SO** E REUSADO EM TODA A EXECUCAO.

ISSO IMPORTA PORQUE, NA PRIMEIRA VEZ, ELE PRECISA GERAR OS EMBEDDINGS DE TODAS AS VIAS, O QUE DEMORA. O RESULTADO FICA GUARDADO EM CACHE NA PASTA `data/cache`, E AS PROXIMAS EXECUCOES CARREGAM DE LA EM SEGUNDOS.

O CACHE E INVALIDADO SOZINHO SE VOCE REPROCESSAR AS VIAS, PORQUE A CHAVE DELE INCLUI UM RESUMO DO CONTEUDO.

SE O `sentence-transformers` NAO ESTIVER INSTALADO, O MOTOR AVISA E CONTINUA SO COM A COMPARACAO DE TEXTO. FUNCIONA, MAS AS CANDIDATAS FICAM PIORES.

In [16]:
# OS VALORES SAO GUARDADOS COMO TEXTO NO BANCO, ENTAO CONVERTEMOS AQUI.
ano = str(ANO)
ibge = str(CODIGO_IBGE)

# CRIA O CLIENTE DA IA UMA VEZ SO. A URL E O QUE APONTA O CLIENTE DA OPENAI
# PARA O DEEPSEEK; O RESTO DO CODIGO NAO SABE QUAL PROVEDOR ESTA ATENDENDO.
client = OpenAI(api_key=config.CHAVE_IA, base_url=config.URL_BASE_IA)
modelo = config.MODELO_IA

conn = conectar()

# ACRESCENTA AS COLUNAS DE ASSOCIACAO E CRIA AS TABELAS DE APOIO.
criar_tabelas(conn)
print("COLUNAS DE ASSOCIACAO E TABELAS DE APOIO PRONTAS.")

# CONFERE QUE EXISTE ACIDENTE REVISADO NESSE RECORTE.
revisados = conn.execute(
    f"SELECT count(*) FROM acidentes_revisado "
    f"WHERE {config.COLUNA_ANO}=? AND {config.COLUNA_MUNICIPIO}=? AND revisao_status='ok'",
    (ano, ibge),
).fetchone()[0]

if revisados == 0:
    raise RuntimeError(
        f"NENHUM ACIDENTE REVISADO PARA ANO={ano}, MUNICIPIO={ibge}.\n"
        "RODE O NOTEBOOK 03_revisao_enderecos.ipynb PARA ESTE RECORTE PRIMEIRO."
    )

print(f"ACIDENTES REVISADOS NO RECORTE: {revisados}")
print()

# CARREGA O MOTOR DE BUSCA. NA PRIMEIRA VEZ ISSO DEMORA, DEPOIS VEM DO CACHE.
print("CARREGANDO O MOTOR DE BUSCA...")
vias = carregar_vias()
motor = MotorBusca(vias, usar_embeddings=config.USAR_EMBEDDINGS)

print()
print(f"VIAS INDEXADAS : {len(motor.vias)}")
print(f"EMBEDDINGS     : {'LIGADOS' if motor.usando_embeddings else 'DESLIGADOS (SO TEXTO)'}")

# VERIFICA SE ESSE RECORTE JA FOI CONCLUIDO EM UMA EXECUCAO ANTERIOR.
cobertura = conn.execute(
    "SELECT status, total_linhas, sem_via FROM associacao_cobertura "
    "WHERE ano=? AND codigo_ibge=?", (ano, ibge)
).fetchone()

RECORTE_JA_CONCLUIDO = bool(cobertura and cobertura[0] == "ok")

print()
if RECORTE_JA_CONCLUIDO:
    print("ESTE RECORTE JA FOI CONCLUIDO EM UMA EXECUCAO ANTERIOR.")
    print(f"   LINHAS  : {cobertura[1]}")
    print(f"   SEM VIA : {cobertura[2]}")
    print()
    print("AS ETAPAS 6 A 8 SERAO PULADAS. VA PARA A ETAPA 9.")
elif cobertura:
    print("ESTE RECORTE ESTA PARCIAL. A EXECUCAO VAI CONTINUAR DE ONDE PAROU.")
else:
    print("RECORTE INEDITO. A EXECUCAO VAI PROCESSAR TUDO.")

COLUNAS DE ASSOCIACAO E TABELAS DE APOIO PRONTAS.
ACIDENTES REVISADOS NO RECORTE: 8865

CARREGANDO O MOTOR DE BUSCA...
[BUSCA] sentence_transformers indisponivel (No module named 'sentence_transformers'). USANDO SO LEXICAL.

VIAS INDEXADAS : 4845
EMBEDDINGS     : DESLIGADOS (SO TEXTO)

ESTE RECORTE ESTA PARCIAL. A EXECUCAO VAI CONTINUAR DE ONDE PAROU.


## 5. EXPERIMENTAR A BUSCA A MAO

ESTA CELULA NAO FAZ PARTE DA PIPELINE. ELA EXISTE PARA VOCE ENTENDER, E MELHORAR, A QUALIDADE DAS CANDIDATAS.

E AQUI QUE MORA O TRABALHO DE PESQUISA DESTE NOTEBOOK. A ASSOCIACAO SO PODE SER TAO BOA QUANTO A LISTA QUE CHEGA AO GEMINI: **SE A VIA CERTA NAO ESTIVER ENTRE AS SEIS CANDIDATAS, NENHUMA INSTRUCAO SALVA A ESCOLHA.**

USE PARA:

- VER O QUE ACONTECE COM UM NOME REPETIDO, COMO `Rua 1`.
- CONFERIR SE UM ENDERECO QUE VOCE SABE QUE EXISTE APARECE EM PRIMEIRO LUGAR.
- COMPARAR O RANKING COM E SEM BAIRRO NA CONSULTA.

PARA MEXER NOS PESOS DA COMBINACAO, EDITE O METODO `buscar` NO ARQUIVO `busca.py` E EXECUTE A CELULA DA ETAPA 4 DE NOVO. OS PESOS ATUAIS SAO:

```text
COM BAIRRO : 0.45 * SEMELHANCA DO NOME + 0.25 * SEMELHANCA DO BAIRRO + 0.30 * SIGNIFICADO
SEM BAIRRO : 0.60 * SEMELHANCA DO NOME + 0.40 * SIGNIFICADO
```

In [17]:
# TROQUE OS VALORES ABAIXO A VONTADE. ESTA CELULA NAO ALTERA NADA NO BANCO.
motor.mostrar(via="Avenida Independencia", bairro="Cidade Livre", limit=6)

print()
print("=" * 70)
print("O CASO DIFICIL: UM NOME QUE SE REPETE NA CIDADE.")
print("REPARE QUE AS PRIMEIRAS CANDIDATAS TEM NOTAS QUASE IDENTICAS.")
print("QUANDO ISSO ACONTECE E O BAIRRO NAO DESEMPATA, A ESCOLHA VIRA SORTEIO.")
print("=" * 70)
print()

motor.mostrar(via="Rua 1", bairro="", limit=6)

CONSULTA: via='Avenida Independencia' bairro='Cidade Livre' | EMBEDDINGS=OFF

  #1  score=0.700  Avenida Independência  [Cidade Livre, Colina Azul, Independência, Independência - 1º Complemento - Setor das Mansões, Jardim Belo Horizonte, Jardim Ipiranga, Jardim Ipiranga - Continuação, Jardim Monte Cristo, Residencial Village Garavelo, Setor Central - Perímetro Urbano, Setor Serra Dourada - 2ª Etapa, Setor Serra Dourada - 3ª Etapa]  5151 m
  #2  score=0.515  Avenida Independência  [Jardim Miramar, Nova Olinda, Nova Olinda - 1º Complemento, Rosa dos Ventos]  1668 m
  #3  score=0.515  Avenida Goiás  [Cidade Livre, Setor Marista Sul]  1051 m
  #4  score=0.499  Avenida Quinze de Novembro  [Cidade Livre, Jardim Monte Cristo]  1624 m
  #5  score=0.493  Avenida Contorno  [Cidade Livre, Jardim Monte Cristo, Setor Marista Sul, Setor Rio Vermelho, Setor dos Estados]  2946 m
  #6  score=0.482  Avenida R3  [Cidade Livre, Colina Azul]  692 m

O CASO DIFICIL: UM NOME QUE SE REPETE NA CIDADE.
REPARE Q

## 6. PASSOS 1 E 2: MARCAR AS LINHAS E LISTAR OS PARES

DOIS PASSOS BARATOS, DENTRO DO BANCO, SEM NENHUMA CHAMADA DE IA.

PRIMEIRO RESOLVEMOS AS LINHAS SEM ENDERECO APROVEITAVEL, QUE VIRAM `sem_via` NA HORA.

DEPOIS MARCAMOS COMO `pendente` AS QUE DE FATO PRECISAM DE BUSCA, E LISTAMOS OS PARES **DISTINTOS** ENTRE ELAS.

A ECONOMIA AQUI E GRANDE PELO MESMO MOTIVO DO NOTEBOOK 03: O MESMO ENDERECO SE REPETE EM DEZENAS OU CENTENAS DE ACIDENTES, E A DECISAO E TOMADA UMA VEZ SO.

In [18]:
if RECORTE_JA_CONCLUIDO:
    print("ETAPA PULADA: O RECORTE JA ESTA CONCLUIDO.")
    pares = []
else:
    # PASSO 1a: RESOLVE AS LINHAS QUE NAO TEM ENDERECO PARA PROCURAR.
    sem_endereco = marcar_sem_endereco(conn, ano, ibge)
    print(f"PASSO 1a  LINHAS SEM ENDERECO -> 'sem_via' : {sem_endereco}")

    # PASSO 1b: MARCA COMO PENDENTE AS QUE PRECISAM DE BUSCA.
    pendentes = marcar_pendentes(conn, ano, ibge)
    print(f"PASSO 1b  LINHAS MARCADAS COMO PENDENTE    : {pendentes}")
    print()

    # PASSO 2: PEGA OS PARES DISTINTOS.
    pares = pares_pendentes(conn, ano, ibge)
    print(f"PASSO 2   PARES DISTINTOS A DECIDIR        : {len(pares)}")

    # SEPARA OS QUE JA ESTAO NO DICIONARIO.
    ineditos = [p for p in pares if not par_conhecido(conn, p[0], p[1])]
    print(f"          JA NO DICIONARIO                 : {len(pares) - len(ineditos)}")
    print(f"          INEDITOS, VAO PARA A IA          : {len(ineditos)}")

    if ineditos:
        lotes_previstos = (len(ineditos) + TAMANHO_LOTE - 1) // TAMANHO_LOTE
        print()
        print(f"SERAO CERCA DE {lotes_previstos} CHAMADAS A IA.")
        print(f"TEMPO MINIMO ESTIMADO: {lotes_previstos * PAUSA_ENTRE_CHAMADAS / 60:.1f} MINUTOS.")
    else:
        print()
        print("NENHUMA CHAMADA A IA SERA NECESSARIA. TUDO VEM DO DICIONARIO.")

PASSO 1a  LINHAS SEM ENDERECO -> 'sem_via' : 0
PASSO 1b  LINHAS MARCADAS COMO PENDENTE    : 0

PASSO 2   PARES DISTINTOS A DECIDIR        : 180
          JA NO DICIONARIO                 : 0
          INEDITOS, VAO PARA A IA          : 180

SERAO CERCA DE 9 CHAMADAS A IA.
TEMPO MINIMO ESTIMADO: 0.1 MINUTOS.


## 7. PASSO 3: BUSCAR AS CANDIDATAS E PEDIR A ESCOLHA

ESTA E A ETAPA DEMORADA E A UNICA QUE CUSTA DINHEIRO.

PARA CADA PAR INEDITO, A BUSCA PROPOE AS CANDIDATAS E O LOTE INTEIRO VAI EM UMA CHAMADA SO A IA.

A GRAVACAO ACONTECE LOTE A LOTE. SE A EXECUCAO FOR INTERROMPIDA NO MEIO, TUDO QUE JA FOI DECIDIDO ESTA SALVO E NAO SERA REFEITO.

A LISTA DE CANDIDATAS OFERECIDA A IA E GRAVADA JUNTO COM A DECISAO. E ESSA COLUNA QUE PERMITE, DEPOIS, ENTENDER POR QUE UMA VIA FOI ESCOLHIDA.

In [19]:
if RECORTE_JA_CONCLUIDO:
    print("ETAPA PULADA: O RECORTE JA ESTA CONCLUIDO.")
    novos_ia = 0
    falhados_ia = 0
elif not pares:
    print("NENHUM PAR PENDENTE. NADA A FAZER.")
    novos_ia = 0
    falhados_ia = 0
else:
    print(f"PROCESSANDO {len(pares)} PARES COM O MODELO {modelo}...")
    print()
    novos_ia, falhados_ia = associar_pares(conn, motor, client, modelo, pares)
    print()
    print(f"DECIDIDOS PELA IA   : {novos_ia}")
    print(f"PENDENTES POR FALHA : {falhados_ia}")

PROCESSANDO 180 PARES COM O MODELO deepseek-v4-flash...


[ASSOCIACAO] LOTE FALHOU APOS 3 TENTATIVAS: RESPOSTA COM 21 ITENS, ESPERADOS 20. SE REPETIR, DIMINUA TAMANHO_LOTE OU AUMENTE MAX_TOKENS_RESPOSTA.
[ASSOCIACAO] ESSES PARES FICAM PENDENTES PARA A PROXIMA EXECUCAO.
160/180 PARES  |  NOVOS NA IA: 140  |  PENDENTES POR FALHA: 20
[ASSOCIACAO] LOTE FALHOU APOS 3 TENTATIVAS: RESPOSTA COM 19 ITENS, ESPERADOS 20. SE REPETIR, DIMINUA TAMANHO_LOTE OU AUMENTE MAX_TOKENS_RESPOSTA.
[ASSOCIACAO] ESSES PARES FICAM PENDENTES PARA A PROXIMA EXECUCAO.
180/180 PARES  |  NOVOS NA IA: 140  |  PENDENTES POR FALHA: 40

DECIDIDOS PELA IA   : 140
PENDENTES POR FALHA : 40


## 8. PASSOS 4 E 5: ESPALHAR O RESULTADO E REGISTRAR A COBERTURA

O `via_id` DECIDIDO PARA CADA PAR E COPIADO PARA TODAS AS LINHAS DE ACIDENTE QUE TEM AQUELE PAR EXATO.

DEPOIS A COBERTURA E ATUALIZADA. O STATUS SO VIRA `ok` QUANDO NENHUMA LINHA DO RECORTE FICOU PENDENTE.

In [20]:
if RECORTE_JA_CONCLUIDO:
    print("ETAPA PULADA: O RECORTE JA ESTA CONCLUIDO.")
else:
    # PASSO 4: COPIA A DECISAO PARA TODAS AS LINHAS DE CADA PAR.
    linhas_com_via, linhas_sem_via = espalhar_resultado(conn, modelo, ano, ibge)
    print(f"LINHAS QUE RECEBERAM UMA VIA : {linhas_com_via}")
    print(f"LINHAS SEM VIA COMPATIVEL    : {linhas_sem_via}")
    print()

    # PASSO 5: REGISTRA O PROGRESSO DESTE RECORTE.
    resumo_cobertura = registrar_cobertura(conn, ano, ibge, len(pares), novos_ia)

    print("COBERTURA DESTE RECORTE:")
    for campo, valor in resumo_cobertura.items():
        print(f"   {campo:16}: {valor}")
    print()

    if resumo_cobertura["status"] == "ok":
        print("RECORTE CONCLUIDO.")
    else:
        print(f"AINDA HA {resumo_cobertura['pendentes']} LINHAS PENDENTES.")
        print("REEXECUTE ESTE NOTEBOOK PARA CONTINUAR DE ONDE PAROU.")

LINHAS QUE RECEBERAM UMA VIA : 184
LINHAS SEM VIA COMPATIVEL    : 56

COBERTURA DESTE RECORTE:
   total           : 8865
   distintos       : 3579
   novos_ia        : 3099
   reaproveitados  : 480
   sem_via         : 2537
   pendentes       : 211
   status          : parcial

AINDA HA 211 LINHAS PENDENTES.
REEXECUTE ESTE NOTEBOOK PARA CONTINUAR DE ONDE PAROU.


## 9. CONFERENCIA DA QUALIDADE

**ESTA E A ETAPA MAIS IMPORTANTE DO NOTEBOOK PARA QUEM VAI USAR OS DADOS.**

O NUMERO DE LINHAS COM `via_id` DIZ QUANTO FOI ASSOCIADO, MAS NAO DIZ QUANTO FOI ASSOCIADO **CORRETAMENTE**. AS TRES MEDIDAS ABAIXO AJUDAM A SEPARAR AS DUAS COISAS.

### 1. DISTRIBUICAO DOS STATUS

QUANTO FICOU `ok`, `sem_via` E `pendente`.

### 2. ASSOCIACOES PARA VIAS DE NOME REPETIDO

ESTE E O NUMERO CRITICO. QUANDO O ACIDENTE FOI VINCULADO A UMA VIA CUJO NOME EXISTE VARIAS VEZES NA CIDADE, A ESCOLHA PODE TER SIDO ARBITRARIA.

O RESULTADO SAI MARCADO COMO `ok` DO MESMO JEITO. NAO EXISTE, NA TABELA, NADA QUE DISTINGA ESSE CASO DE UM ACERTO SEGURO. POR ISSO ELE PRECISA SER MEDIDO AQUI.

### 3. AMOSTRA PARA CONFERENCIA MANUAL

OS PARES COM MENOR NOTA. SE A ASSOCIACAO VAI ERRAR, E MAIS PROVAVEL QUE ERRE AQUI. VALE OLHAR ALGUNS A OLHO.

In [21]:
filtro = (f"WHERE {config.COLUNA_ANO}='{ano}' AND {config.COLUNA_MUNICIPIO}='{ibge}' "
          f"AND revisao_status='ok'")

# 1. DISTRIBUICAO DOS STATUS.
print("1. DISTRIBUICAO DOS STATUS")
print()
distribuicao = pd.read_sql_query(
    f"SELECT COALESCE(assoc_status, 'nao processado') AS status, count(*) AS linhas "
    f"FROM acidentes_revisado {filtro} GROUP BY 1 ORDER BY linhas DESC",
    conn,
)
total_linhas = int(distribuicao["linhas"].sum())
distribuicao["percentual"] = (distribuicao["linhas"] / total_linhas * 100).round(1)
print(distribuicao.to_string(index=False))

# 2. QUANTO FOI PARA VIA DE NOME REPETIDO.
print()
print("=" * 70)
print("2. ASSOCIACOES PARA VIAS DE NOME REPETIDO")
print()

nomes_repetidos = (
    "SELECT nome_via FROM vias_processadas GROUP BY nome_via HAVING count(*) > 1"
)

com_via = conn.execute(
    f"SELECT count(*) FROM acidentes_revisado {filtro} AND assoc_status='ok'"
).fetchone()[0]

em_nome_repetido = conn.execute(
    f"SELECT count(*) FROM acidentes_revisado {filtro} AND assoc_status='ok' "
    f"AND nome_via_associada IN ({nomes_repetidos})"
).fetchone()[0]

print(f"LINHAS COM VIA ASSOCIADA           : {com_via}")
if com_via:
    print(f"   DESSAS, PARA NOME REPETIDO      : {em_nome_repetido}  "
          f"({em_nome_repetido / com_via * 100:.1f}%)")
    print()
    print("ESSE PERCENTUAL E O SEU LIMITE DE CONFIANCA. NELE, A VIA CERTA E UMA")
    print("ENTRE VARIAS DE MESMO NOME, E O DESEMPATE DEPENDEU DE UM BAIRRO QUE")
    print("NEM SEMPRE EXISTE. TRATE ESSAS LINHAS COMO INCERTAS.")

    print()
    print("   AS VIAS DE NOME REPETIDO QUE MAIS RECEBERAM ACIDENTES:")
    for nome, quantidade in conn.execute(
        f"SELECT nome_via_associada, count(*) c FROM acidentes_revisado {filtro} "
        f"AND assoc_status='ok' AND nome_via_associada IN ({nomes_repetidos}) "
        f"GROUP BY 1 ORDER BY c DESC LIMIT 8"
    ):
        homonimas = conn.execute(
            "SELECT count(*) FROM vias_processadas WHERE nome_via=?", (nome,)
        ).fetchone()[0]
        print(f"      {quantidade:5} acidentes -> '{nome}'  ({homonimas} vias com esse nome)")

# 3. AMOSTRA COM AS MENORES NOTAS.
print()
print("=" * 70)
print("3. AS 15 ASSOCIACOES DE MENOR NOTA (CONFIRA ESTAS A OLHO)")
print()

amostra = pd.read_sql_query(
    f"SELECT DISTINCT end_acidente_padronizado AS endereco, "
    f"       bairro_acidente_padronizado AS bairro, "
    f"       nome_via_associada AS via_escolhida, "
    f"       ROUND(assoc_score, 3) AS nota "
    f"FROM acidentes_revisado {filtro} AND assoc_status='ok' "
    f"ORDER BY assoc_score ASC LIMIT 15",
    conn,
)
print(amostra.to_string(index=False))

conn.close()

1. DISTRIBUICAO DOS STATUS

  status  linhas  percentual
      ok    6117        69.0
 sem_via    2537        28.6
pendente     211         2.4

2. ASSOCIACOES PARA VIAS DE NOME REPETIDO

LINHAS COM VIA ASSOCIADA           : 6117
   DESSAS, PARA NOME REPETIDO      : 2823  (46.2%)

ESSE PERCENTUAL E O SEU LIMITE DE CONFIANCA. NELE, A VIA CERTA E UMA
ENTRE VARIAS DE MESMO NOME, E O DESEMPATE DEPENDEU DE UM BAIRRO QUE
NEM SEMPRE EXISTE. TRATE ESSAS LINHAS COMO INCERTAS.

   AS VIAS DE NOME REPETIDO QUE MAIS RECEBERAM ACIDENTES:
        193 acidentes -> 'Rodovia GO-040'  (2 vias com esse nome)
        167 acidentes -> 'Avenida São Paulo'  (2 vias com esse nome)
        162 acidentes -> 'Avenida Independência'  (2 vias com esse nome)
        132 acidentes -> 'Avenida Liberdade'  (2 vias com esse nome)
         96 acidentes -> 'Avenida Brasil'  (6 vias com esse nome)
         90 acidentes -> 'Avenida São João'  (3 vias com esse nome)
         72 acidentes -> 'Avenida Bela Vista'  (5 vias com

## 10. VISAO GERAL DE TODOS OS RECORTES

ESTA CELULA PODE SER EXECUTADA A QUALQUER MOMENTO, INDEPENDENTE DO RESTO DO NOTEBOOK.

ELA MOSTRA O MAPA COMPLETO DO QUE JA FOI ASSOCIADO, POR ANO E MUNICIPIO.

In [22]:
conexao = sqlite3.connect(str(config.BANCO))

try:
    cobertura_geral = pd.read_sql_query(
        "SELECT ano, codigo_ibge, status, total_linhas, distintos, novos_ia, "
        "       reaproveitados, sem_via, concluido_em "
        "FROM associacao_cobertura ORDER BY ano DESC, codigo_ibge",
        conexao,
    )

    if cobertura_geral.empty:
        print("NENHUM RECORTE ASSOCIADO AINDA.")
    else:
        print("COBERTURA DA ASSOCIACAO:")
        print()
        print(cobertura_geral.to_string(index=False))

        total_distintos = int(cobertura_geral["distintos"].sum())
        total_ia = int(cobertura_geral["novos_ia"].sum())

        if total_distintos:
            print()
            print(f"PARES DISTINTOS PROCESSADOS : {total_distintos}")
            print(f"ENVIADOS A IA               : {total_ia}")
            print(f"ECONOMIA PELO DICIONARIO    : {(1 - total_ia / total_distintos) * 100:.1f}%")
finally:
    conexao.close()

COBERTURA DA ASSOCIACAO:

 ano codigo_ibge  status  total_linhas  distintos  novos_ia  reaproveitados  sem_via                     concluido_em
2024     5201405 parcial          8865       3579      3099             480     2537 2026-08-13T19:33:54.986225+00:00

PARES DISTINTOS PROCESSADOS : 3579
ENVIADOS A IA               : 3099
ECONOMIA PELO DICIONARIO    : 13.4%


## FIM DA SEQUENCIA

O OBJETIVO DO PROJETO ESTA CUMPRIDO PARA ESTE RECORTE: CADA ACIDENTE TEM, NA COLUNA `via_id_associada`, A VIA A QUE ELE PERTENCE.

PARA PROCESSAR OUTRO MUNICIPIO, TROQUE `ANO` E `CODIGO_IBGE` NA ETAPA 1 DOS NOTEBOOKS 03 E 04 E EXECUTE OS DOIS DE NOVO.

## DUAS COISAS PARA LEMBRAR

**A TABELA DE VIAS E DE UMA CIDADE SO.** SE VOCE PROCESSAR UM MUNICIPIO DIFERENTE DAQUELE QUE ESTA EM `vias_processadas`, OS ACIDENTES SERAO ASSOCIADOS A RUAS DA CIDADE ERRADA, COM NOMES QUE COINCIDEM. O RESULTADO SAI MARCADO COMO `ok` E NADA AVISA.

**REPROCESSAR AS VIAS INVALIDA OS VINCULOS.** O `via_id` DEPENDE DA GEOMETRIA. SE VOCE RODAR O NOTEBOOK 02 DE NOVO E O OPENSTREETMAP TIVER MUDADO, OS IDENTIFICADORES MUDAM E ESTES VINCULOS PASSAM A APONTAR PARA VIAS QUE NAO EXISTEM MAIS. REPROCESSOU A MALHA, REPROCESSE TAMBEM A ASSOCIACAO.